<a href="https://colab.research.google.com/github/bogomdanalnur135-coder/NU-/blob/main/slss_multimodal_micronet_resnet18_v2_(S0_S27%2C_N%3D28).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multimodal SLSS Prediction: Frozen MicroNet ResNet18 + Tabular Parameters

Predict SLSS (MPa) of KCl/DMF-anodized Al2024-T3/PU single-lap joints
(Bakhbergen et al., Smart Mater. Manuf. 3 (2025) 100100) from tabular process
parameters + surface measurements + AFM 3D renders + SEM cross/surface images,
all in one LOOCV pipeline.

**Samples.** S0-S27 (28 samples; 3 concentrations x 3 voltages x 3 times full
factorial + S0 untreated). S0 is kept as the 0 M / 0 V / 0 min anchor
(INCLUDE_S0 = True); set it to False to reproduce an S1-S27 run.

**Tabular data.** Built from the RAW replicate measurements of KCl_ver_3_data.xlsx
(3 per sample for roughness, valley depth, SLSS, fracture energy), averaged in
this script. Each mean is asserted to lie between the min and max replicate and
is compared with the (rounded) Table 2 of the paper. The paper/Excel 'AVG' cells
of S2, S4, S9, S10, S11, S13, S19, S26, S27 average only 2 of 3 replicates
(range F:G instead of E:G); here all 3 are used (USE_ALL_REPLICATES = True).

**Method.** ResNet18 pretrained on MicroNet, **frozen** (no fine-tuning).
Each image -> 512-dim embedding, averaged per sample. Per fold: MinMaxScaler +
PCA fit on training fold only; image blocks concatenated with tabular features;
8 regressors; feature-set configurations below. Metrics: MAPE, RMSE, R2.

**Key question.** Do images add signal beyond tabular parameters?
(`FS3_tabular_only` vs each `tabular_plus_*`, same PCA, model by model.)

**Rules.** PCA inside each fold; sensitivity for n_pca in {2,3,5}, primary
n_pca=5. `SEM_top_view_failure` excluded (target leakage); `_cropped` /
`_highlight` excluded.

**AFM note.** AFM files are rendered 3D images, not height maps - calibrated
Ra/Rq cannot be recovered. Processed via the same CNN pathway as SEM.

**Outputs.** `full_multimodal_results.csv`, `pca_sensitivity_results.csv`,
`tabular_vs_multimodal_delta.csv`, `image_counts_per_sample.csv`.

**Caveat N=28.** A few MAPE points can be noise; claims must survive both
the PCA-sensitivity and per-model delta tables.
"""

In [1]:
# =============================================================================
# 0. INSTALL (paste this whole file into ONE Colab cell)
# =============================================================================
get_ipython().system('pip -q install segmentation-models-pytorch torch torchvision pillow')

# =============================================================================
# 1. IMPORTS
# =============================================================================
import os
import re
import zipfile
import shutil
import warnings

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torchvision import transforms
from segmentation_models_pytorch.encoders import get_encoder, encoders

from sklearn.base import clone
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor,
                               BaggingRegressor, GradientBoostingRegressor,
                               AdaBoostRegressor)
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Libraries loaded.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 2.3 MB/s eta 0:00:00
Libraries loaded.


In [2]:
# =============================================================================
# 2. UPLOAD ZIP
# =============================================================================
from google.colab import files

print("=" * 80)
print("UPLOAD YOUR ZIP (AFM_3_D / SEM_cross_section / SEM_Surface, images for S0-S27 - an extra")
print("top-level folder is fine). SEM_top_view_failure will be ignored if present.")
print("=" * 80)
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No file was uploaded.")
zip_candidates = [n for n in uploaded if n.lower().endswith(".zip")]
if not zip_candidates:
    raise RuntimeError("No .zip file found among uploaded files.")
ZIP_PATH = zip_candidates[0]
print(f"Uploaded: {ZIP_PATH}")

UPLOAD YOUR ZIP (AFM_3_D / SEM_cross_section / SEM_Surface, images for S0-S27 - an extra
top-level folder is fine). SEM_top_view_failure will be ignored if present.


Saving images.zip to images.zip
Uploaded: images.zip


In [3]:
# =============================================================================
# 3. EXTRACT
# =============================================================================
EXTRACT_ROOT = "/content/image_dataset"
if os.path.exists(EXTRACT_ROOT):
    shutil.rmtree(EXTRACT_ROOT)
os.makedirs(EXTRACT_ROOT, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_ROOT)
print(f"Extracted to {EXTRACT_ROOT}")

Extracted to /content/image_dataset


In [4]:
# =============================================================================
# 4. JUNK FILTER (__MACOSX, .DS_Store, AppleDouble "._xxx")
# =============================================================================
JUNK_DIR_NAMES = {"__MACOSX"}
IMAGE_EXTS = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".webp")

def is_junk_file(filename: str) -> bool:
    base = os.path.basename(filename)
    return base.startswith("._") or base == ".DS_Store" or not base.lower().endswith(IMAGE_EXTS)

# Filenames like "..._cropped.tif" / "..._highlight.tif" are edited copies of
# an original image (not an independent view) - excluded from the PRIMARY run.
EXCLUDE_VARIANT_KEYWORDS = ("cropped", "highlight")

def is_excluded_variant(filename: str) -> bool:
    base = os.path.basename(filename).lower()
    return any(k in base for k in EXCLUDE_VARIANT_KEYWORDS)

MAG_RE = re.compile(r"(\d+)\s*k(?=[_.\-]|$)", re.IGNORECASE)

def parse_magnification(filename: str):
    m = MAG_RE.search(os.path.basename(filename))
    return f"{m.group(1)}k" if m else None

def walk_clean(root):
    for current_root, dirs, filenames in os.walk(root):
        dirs[:] = [d for d in dirs if d not in JUNK_DIR_NAMES]
        yield current_root, dirs, filenames

In [5]:
# =============================================================================
# 5. FIND REQUIRED DIRECTORIES (robust to an extra top-level folder)
# =============================================================================
def find_directory(root, target_name):
    matches = []
    for current_root, dirs, _ in walk_clean(root):
        for d in dirs:
            if d.lower() == target_name.lower():
                matches.append(os.path.join(current_root, d))
    return matches

def find_directory_flexible(root, possible_names):
    for name in possible_names:
        m = find_directory(root, name)
        if m:
            return m[0]
    return None

def find_subdir_by_keywords(root, keywords):
    """Fallback only - used when the exact expected folder name isn't found."""
    candidates = []
    for d in os.listdir(root):
        full = os.path.join(root, d)
        if not os.path.isdir(full) or d in JUNK_DIR_NAMES:
            continue
        score = sum(1 for k in keywords if k.lower() in d.lower())
        if score > 0:
            candidates.append((score, full))
    candidates.sort(reverse=True)
    return candidates[0][1] if candidates else None

AFM_ROOT = find_directory_flexible(EXTRACT_ROOT, ["AFM_3_D", "AFM_3D", "AFM"])
SEM_CROSS_ROOT = find_directory_flexible(EXTRACT_ROOT, ["SEM_cross_section", "SEM_Cross_Section"])
SEM_SURFACE_ROOT = find_directory_flexible(EXTRACT_ROOT, ["SEM_Surface", "SEM_surface"])
FAILURE_ROOT = find_directory_flexible(EXTRACT_ROOT, ["SEM_top_view_failure", "SEM_top_failure", "top_view_failure"])

print("\nDIRECTORIES FOUND")
print("AFM root:        ", AFM_ROOT)
print("SEM cross root:  ", SEM_CROSS_ROOT)
print("SEM surface root:", SEM_SURFACE_ROOT)
if FAILURE_ROOT:
    print(f"\nNOTE: found {FAILURE_ROOT} - NOT loaded (post-fracture image = target leakage).")

if AFM_ROOT is None:
    raise FileNotFoundError("Could not find AFM_3_D folder.")
if SEM_CROSS_ROOT is None:
    raise FileNotFoundError("Could not find SEM_cross_section folder.")
if SEM_SURFACE_ROOT is None:
    raise FileNotFoundError("Could not find SEM_Surface folder.")

# Exact names first (we know what they actually are), fuzzy only as fallback.
AFM_3UM_ROOT = find_directory_flexible(AFM_ROOT, ["3_3_micron", "3x3_micron"])
if AFM_3UM_ROOT is None:
    print("WARNING: exact '3_3_micron' not found, falling back to fuzzy match.")
    AFM_3UM_ROOT = find_subdir_by_keywords(AFM_ROOT, ["3", "micron"])

AFM_10UM_ROOT = find_directory_flexible(AFM_ROOT, ["10_10_micron", "10x10_micron"])
if AFM_10UM_ROOT is None:
    print("WARNING: exact '10_10_micron' not found, falling back to fuzzy match.")
    AFM_10UM_ROOT = find_subdir_by_keywords(AFM_ROOT, ["10", "micron"])

print("\nAFM 3 micron folder: ", AFM_3UM_ROOT)
print("AFM 10 micron folder:", AFM_10UM_ROOT)
if AFM_3UM_ROOT is None or AFM_10UM_ROOT is None:
    raise FileNotFoundError("Could not identify AFM 3um / 10um subfolders inside AFM_3_D.")


DIRECTORIES FOUND
AFM root:         /content/image_dataset/images/AFM_3D
SEM cross root:   /content/image_dataset/images/SEM_cross_section
SEM surface root: /content/image_dataset/images/SEM_Surface

NOTE: found /content/image_dataset/images/SEM_top_view_failure - NOT loaded (post-fracture image = target leakage).

AFM 3 micron folder:  /content/image_dataset/images/AFM_3D/3_3_micron
AFM 10 micron folder: /content/image_dataset/images/AFM_3D/10_10_micron


In [6]:
# =============================================================================
# 6. SAMPLE INDEX PARSER
# =============================================================================
IDX_RE = re.compile(r"^[Ss]?(\d+)(?=[_\-.\s]|$)")      # "12_xxx", "S12_xxx", "12"
CODE_RE = re.compile(r"^(\d+-\d+-\d+)")                  # Excel 'Code' column, e.g. "01-12-15"

# Excel 'Code' column -> sample index. Only used if image files are named by
# that code (checked BEFORE the numeric-prefix rule, otherwise "005-8-5" would
# be misread as sample 5).
CODE_TO_IDX = {"0-0-0": 0, "005-8-5": 1, "005-10-5": 2, "005-12-5": 3, "0075-8-5": 4, "0075-10-5": 5, "0075-12-5": 6, "01-8-5": 7, "01-10-5": 8, "01-12-5": 9, "005-8-10": 10, "005-10-10": 11, "005-12-10": 12, "0075-8-10": 13, "0075-10-10": 14, "0075-12-10": 15, "01-8-10": 16, "01-10-10": 17, "01-12-10": 18, "005-8-15": 19, "005-10-15": 20, "005-12-15": 21, "0075-8-15": 22, "0075-10-15": 23, "0075-12-15": 24, "01-8-15": 25, "01-10-15": 26, "01-12-15": 27}

def parse_sample_index(name: str):
    base = os.path.basename(name)
    mc = CODE_RE.match(base)
    if mc and mc.group(1) in CODE_TO_IDX:
        return CODE_TO_IDX[mc.group(1)]
    m = IDX_RE.match(base)
    return int(m.group(1)) if m else None

INCLUDE_S0 = True    # S0 = untreated reference (0 M, 0 V, 0 min); images 0-27
EXPECTED_SAMPLES = list(range(0 if INCLUDE_S0 else 1, 28))   # S0-S27 (or S1-S27)

In [7]:
# =============================================================================
# 7. IMAGE COLLECTORS (variant-aware, magnification-aware)
# =============================================================================
def collect_images_flat(root, exclude_variants=False):
    result, unmatched, excluded = {}, [], []
    for f in sorted(os.listdir(root)):
        full = os.path.join(root, f)
        if not os.path.isfile(full) or is_junk_file(f):
            continue
        if exclude_variants and is_excluded_variant(f):
            excluded.append(full)
            continue
        idx = parse_sample_index(f)
        if idx is None:
            unmatched.append(f)
            continue
        result.setdefault(idx, []).append(full)
    return result, unmatched, excluded

def collect_images_by_sample_folder(root, exclude_variants=False):
    result, unmatched, excluded = {}, [], []
    for entry in sorted(os.listdir(root)):
        full = os.path.join(root, entry)
        if not os.path.isdir(full) or entry in JUNK_DIR_NAMES:
            continue
        idx = parse_sample_index(entry)
        files = []
        for f in os.listdir(full):
            fpath = os.path.join(full, f)
            if not os.path.isfile(fpath) or is_junk_file(f):
                continue
            if exclude_variants and is_excluded_variant(f):
                excluded.append(fpath)
                continue
            files.append(fpath)
        if idx is None:
            unmatched.extend(files)
            continue
        result.setdefault(idx, []).extend(files)
    return result, unmatched, excluded

# AFM: no cropped/highlight variants known to exist there, but the filter is
# harmless if they ever do.
afm_3um_images, afm_3um_unmatched, afm_3um_excluded = collect_images_flat(AFM_3UM_ROOT, exclude_variants=True)
afm_10um_images, afm_10um_unmatched, afm_10um_excluded = collect_images_flat(AFM_10UM_ROOT, exclude_variants=True)
sem_cross_images, sem_cross_unmatched, sem_cross_excluded = collect_images_by_sample_folder(SEM_CROSS_ROOT, exclude_variants=True)
sem_surface_images, sem_surface_unmatched, sem_surface_excluded = collect_images_by_sample_folder(SEM_SURFACE_ROOT, exclude_variants=True)

In [8]:
# =============================================================================
# 8. PRE-FLIGHT REPORT
# =============================================================================
def report(name, mapping, unmatched, excluded):
    print(f"\n--- {name} ---")
    for idx in EXPECTED_SAMPLES:
        n = len(mapping.get(idx, []))
        mags = sorted({parse_magnification(p) for p in mapping.get(idx, []) if parse_magnification(p)})
        mag_str = f" [{', '.join(mags)}]" if mags else ""
        print(f"  S{idx:02d}: {'MISSING' if n == 0 else f'{n} image(s){mag_str}'}")
    if excluded:
        print(f"  Excluded as cropped/highlight variants: {len(excluded)} file(s) "
              f"(e.g. {os.path.basename(excluded[0])})")
    if unmatched:
        print(f"  Skipped, no parseable sample index: {len(unmatched)} file(s) "
              f"(e.g. {os.path.basename(unmatched[0])})")

print("\n" + "=" * 80)
print("IMAGE DATASET PRE-FLIGHT CHECK")
print("=" * 80)
report("AFM 3x3 um", afm_3um_images, afm_3um_unmatched, afm_3um_excluded)
report("AFM 10x10 um", afm_10um_images, afm_10um_unmatched, afm_10um_excluded)
report("SEM CROSS SECTION", sem_cross_images, sem_cross_unmatched, sem_cross_excluded)
report("SEM SURFACE", sem_surface_images, sem_surface_unmatched, sem_surface_excluded)

def check_all_samples(name, mapping):
    missing = [i for i in EXPECTED_SAMPLES if not mapping.get(i)]
    if missing:
        raise RuntimeError(f"{name} is missing: {['S%d' % i for i in missing]}. "
                            f"Fix the ZIP structure before continuing.")

check_all_samples("AFM 3x3", afm_3um_images)
check_all_samples("AFM 10x10", afm_10um_images)
check_all_samples("SEM cross-section", sem_cross_images)
check_all_samples("SEM surface", sem_surface_images)
print(f"\nALL S{EXPECTED_SAMPLES[0]}-S{EXPECTED_SAMPLES[-1]} PRESENT (after excluding cropped/highlight variants). Pre-flight check passed.")


IMAGE DATASET PRE-FLIGHT CHECK

--- AFM 3x3 um ---
  S00: 1 image(s)
  S01: 1 image(s)
  S02: 1 image(s)
  S03: 1 image(s)
  S04: 1 image(s)
  S05: 1 image(s)
  S06: 1 image(s)
  S07: 1 image(s)
  S08: 1 image(s)
  S09: 1 image(s)
  S10: 1 image(s)
  S11: 1 image(s)
  S12: 1 image(s)
  S13: 1 image(s)
  S14: 1 image(s)
  S15: 1 image(s)
  S16: 1 image(s)
  S17: 1 image(s)
  S18: 1 image(s)
  S19: 1 image(s)
  S20: 1 image(s)
  S21: 1 image(s)
  S22: 1 image(s)
  S23: 1 image(s)
  S24: 1 image(s)
  S25: 1 image(s)
  S26: 1 image(s)
  S27: 1 image(s)

--- AFM 10x10 um ---
  S00: 1 image(s)
  S01: 1 image(s)
  S02: 1 image(s)
  S03: 1 image(s)
  S04: 1 image(s)
  S05: 1 image(s)
  S06: 1 image(s)
  S07: 1 image(s)
  S08: 1 image(s)
  S09: 1 image(s)
  S10: 1 image(s)
  S11: 1 image(s)
  S12: 1 image(s)
  S13: 1 image(s)
  S14: 1 image(s)
  S15: 1 image(s)
  S16: 1 image(s)
  S17: 1 image(s)
  S18: 1 image(s)
  S19: 1 image(s)
  S20: 1 image(s)
  S21: 1 image(s)
  S22: 1 image(s)
  S23: 1

In [9]:
# =============================================================================
# 9. TABULAR DATA (S0-S27) LOADED FROM THE EXCEL FILE + FEATURE SETS
#    Source: KCl_ver_3_data.xlsx. Raw replicates are read from the sheets
#    'Roughness', 'Valley depth', 'Lap shear strength', 'Fracture energy',
#    'Surface free energy' and averaged here (the Excel AVG cells are only used
#    for the comparison check). Nothing is typed in by hand.
# =============================================================================
TABLE_XLSX = "KCl_ver_3_data.xlsx"      # file name in the Colab working dir / Drive

if not os.path.exists(TABLE_XLSX):
    try:                                 # Colab: pick the file in the upload dialog
        from google.colab import files
        print(f"{TABLE_XLSX} not found - upload it now:")
        up = files.upload()
        TABLE_XLSX = next((k for k in up if k.lower().endswith((".xlsx", ".xlsm"))), TABLE_XLSX)
    except ImportError:
        pass
assert os.path.exists(TABLE_XLSX), (
    f"Excel table '{TABLE_XLSX}' not found. Upload it or set TABLE_XLSX to its path.")

USE_ALL_REPLICATES = True     # False -> mimic the 2-of-3 AVG cells of the original Excel
FE_UNIT_FACTOR = 10.0         # Excel fracture energy x10 = paper Table 2 (mJ/mm3)

# sheet -> (column positions of replicates 1-3, position of the Excel AVG cell, short name)
# Positions are counted AFTER 'Sample' becomes the index, i.e. 0-2 = Conc/Voltage/Time,
# 3-5 = replicates 1-3, 6 = Excel AVG (positions, not header names: headers have typos).
SHEETS = {
    "Roughness":          dict(rep=[3, 4, 5], avg=6,  key="Ra"),
    "Valley depth":       dict(rep=[3, 4, 5], avg=6,  key="VD"),
    "Lap shear strength": dict(rep=[3, 4, 5], avg=6,  key="SLSS"),
    "Fracture energy":    dict(rep=[3, 4, 5], avg=6,  key="FE"),
}
SAMPLE_IDS = [f"S{i}" for i in range(28)]

def read_sheet(name):
    df = pd.read_excel(TABLE_XLSX, sheet_name=name)
    df = df.rename(columns={df.columns[0]: "Sample"})
    df["Sample"] = df["Sample"].astype(str).str.strip()
    df = df[df["Sample"].isin(SAMPLE_IDS)].drop_duplicates("Sample").set_index("Sample")
    missing = [s for s in SAMPLE_IDS if s not in df.index]
    assert not missing, f"Sheet '{name}': missing samples {missing}"
    return df.loc[SAMPLE_IDS]

sheets = {n: read_sheet(n) for n in list(SHEETS) + ["Surface free energy"]}

# Conc / Voltage / Time must be identical in every sheet
cond = sheets["Roughness"].iloc[:, 0:3].astype(float)
cond.columns = ["Concentration_M", "Voltage_V", "Time_min"]
for n, df in sheets.items():
    other = df.iloc[:, 0:3].astype(float).values
    assert np.allclose(other, cond.values), f"Sheet '{n}': Conc/Voltage/Time differ from 'Roughness'"

raw = cond.copy()
excel_avg = {}
for n, cfg in SHEETS.items():
    df = sheets[n]
    for k, c in enumerate(cfg["rep"], start=1):
        raw[f"{cfg['key']}_{k}"] = pd.to_numeric(df.iloc[:, c], errors="raise")
    excel_avg[cfg["key"]] = pd.to_numeric(df.iloc[:, cfg["avg"]], errors="coerce")
raw["SurfaceFreeEnergy_mJm2"] = pd.to_numeric(sheets["Surface free energy"].iloc[:, 3], errors="raise")
assert not raw.isna().any().any(), "NaN in raw replicates - check the Excel file"
# Column-position sanity check: the roughness AVG cells use all 3 replicates, so
# they must equal the mean of the 3 replicates that were just read.
assert np.allclose(excel_avg["Ra"], rep_cols_tmp := raw[["Ra_1", "Ra_2", "Ra_3"]].mean(axis=1), atol=1e-6), \
    "Column positions are wrong: roughness replicates do not reproduce the Excel AVG column"

def rep_cols(prefix):
    return raw[[f"{prefix}_1", f"{prefix}_2", f"{prefix}_3"]]

# Excel cells that average only replicates 2-3 (formula range F:G instead of E:G)
PARTIAL_AVG_SAMPLES = ["S2", "S4", "S9", "S10", "S11", "S13", "S19", "S26", "S27"]
PARTIAL_AVG_PROPS = ["VD", "SLSS", "FE"]      # Roughness AVG cells are all 3-replicate

def replicate_mean(prefix):
    df = rep_cols(prefix)
    full = df.mean(axis=1)
    if USE_ALL_REPLICATES or prefix not in PARTIAL_AVG_PROPS:
        return full
    out = full.copy()
    for s in PARTIAL_AVG_SAMPLES:
        out[s] = df.loc[s, [f"{prefix}_2", f"{prefix}_3"]].mean()
    return out

data = pd.DataFrame({
    "Concentration_M":       raw["Concentration_M"],
    "Voltage_V":             raw["Voltage_V"],
    "Time_min":              raw["Time_min"],
    "Roughness_nm":          replicate_mean("Ra"),
    "ValleyDepth_nm":        replicate_mean("VD"),
    "SurfaceFreeEnergy_mJm2": raw["SurfaceFreeEnergy_mJm2"],
    "SLSS_MPa":              replicate_mean("SLSS"),
    "FractureEnergy_mJmm3":  replicate_mean("FE") * FE_UNIT_FACTOR,
})

# --- Check 1: every mean must lie between the min and max replicate -----------
for prefix, col, k in [("Ra", "Roughness_nm", 1), ("VD", "ValleyDepth_nm", 1),
                       ("SLSS", "SLSS_MPa", 1), ("FE", "FractureEnergy_mJmm3", FE_UNIT_FACTOR)]:
    rc = rep_cols(prefix) * k
    ok = (data[col] >= rc.min(axis=1) - 1e-9) & (data[col] <= rc.max(axis=1) + 1e-9)
    assert ok.all(), f"{col}: mean outside replicate range for {list(ok[~ok].index)}"
print("Check 1 passed: every averaged value lies between its min and max replicate.")

# --- Check 2: compare with the AVG cells stored in the Excel itself -----------
# (these are the numbers printed in Table 2 of the paper)
cmp_cols = {"Roughness_nm": ("Ra", 1.0), "ValleyDepth_nm": ("VD", 1.0),
            "SLSS_MPa": ("SLSS", 1.0), "FractureEnergy_mJmm3": ("FE", FE_UNIT_FACTOR)}
dev_pct = pd.DataFrame({col: (excel_avg[k] * f - data[col]) / data[col] * 100
                        for col, (k, f) in cmp_cols.items()})
print("\nMax |deviation| of the Excel AVG cells (= paper Table 2) from replicate means (%):")
print(dev_pct.abs().max().round(3).to_string())
flagged = dev_pct.abs().stack()
flagged = flagged[flagged > 1.0]
if len(flagged):
    print("\nDeviations > 1% (Excel AVG cell used only 2 of 3 replicates):")
    print(flagged.round(2).to_string())

ml_data = data.copy() if INCLUDE_S0 else data.drop(index="S0")
ml_data = ml_data.loc[[f"S{i}" for i in EXPECTED_SAMPLES]]
TARGET = "SLSS_MPa"        # alternative: "FractureEnergy_mJmm3"

feature_sets = {
    "FS1_process_only":              ["Concentration_M", "Voltage_V", "Time_min"],
    "FS2_process_plus_SFE":          ["Concentration_M", "Voltage_V", "Time_min",
                                      "SurfaceFreeEnergy_mJm2"],
    "FS3_process_plus_Roughness_SFE": ["Concentration_M", "Voltage_V", "Time_min",
                                      "Roughness_nm", "SurfaceFreeEnergy_mJm2"],
    "FS4_process_plus_Ra_VD_SFE":    ["Concentration_M", "Voltage_V", "Time_min",
                                      "Roughness_nm", "ValleyDepth_nm",
                                      "SurfaceFreeEnergy_mJm2"],
}
y = ml_data[TARGET]
X_tab = ml_data[feature_sets["FS3_process_plus_Roughness_SFE"]]   # primary tabular baseline
print(f"\nTabular data: N={len(ml_data)} ({ml_data.index[0]}-{ml_data.index[-1]}), target={TARGET}")
assert [f"S{i}" for i in EXPECTED_SAMPLES] == list(ml_data.index)

KCl_ver_3_data.xlsx not found - upload it now:


Saving KCl_ver_3_data.xlsx to KCl_ver_3_data.xlsx
Check 1 passed: every averaged value lies between its min and max replicate.

Max |deviation| of the Excel AVG cells (= paper Table 2) from replicate means (%):
Roughness_nm             0.000
ValleyDepth_nm           8.754
SLSS_MPa                 9.729
FractureEnergy_mJmm3    29.947

Deviations > 1% (Excel AVG cell used only 2 of 3 replicates):
Sample                      
S2      ValleyDepth_nm           1.39
        SLSS_MPa                 2.03
        FractureEnergy_mJmm3    13.28
S4      ValleyDepth_nm           8.75
        SLSS_MPa                 1.86
        FractureEnergy_mJmm3    11.59
S9      ValleyDepth_nm           4.74
        SLSS_MPa                 9.73
        FractureEnergy_mJmm3    20.42
S10     ValleyDepth_nm           5.62
        SLSS_MPa                 1.35
        FractureEnergy_mJmm3    13.68
S11     ValleyDepth_nm           2.39
        SLSS_MPa                 1.31
        FractureEnergy_mJmm3     3.63
S13

In [10]:
# =============================================================================
# 10. MODELS
# =============================================================================
def build_models(random_state=RANDOM_STATE):
    return {
        "DT": DecisionTreeRegressor(random_state=random_state),
        "RF": RandomForestRegressor(n_estimators=100, max_features=1, random_state=random_state),
        "ET": ExtraTreesRegressor(n_estimators=100, max_features=1, random_state=random_state),
        "BG": BaggingRegressor(n_estimators=10, random_state=random_state),
        "GB": GradientBoostingRegressor(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=random_state),
        "AB": AdaBoostRegressor(n_estimators=50, learning_rate=1.0, random_state=random_state),
        "SVR": SVR(),
        "KNN": KNeighborsRegressor(),
    }

In [11]:
# =============================================================================
# 11. FROZEN MICRONET-PRETRAINED RESNET18
# =============================================================================

print("\n" + "=" * 80)
print("LOADING MICRONET-PRETRAINED RESNET18 (frozen)")
print("=" * 80)

import requests
import torch
import torchvision

from torchvision import models, transforms

# -------------------------------------------------------------------------
# 11.1 Device
# -------------------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# -------------------------------------------------------------------------
# 11.2 Download MicroNet checkpoint
# -------------------------------------------------------------------------
MICRONET_URL = (
    "https://huggingface.co/jstuckner/"
    "microscopy-resnet18-micronet/resolve/main/"
    "resnet18_micronet_weights.pth"
)

MICRONET_PATH = "/content/resnet18_micronet_weights.pth"

if not os.path.exists(MICRONET_PATH):
    print("Downloading MicroNet ResNet18 checkpoint...")

    response = requests.get(MICRONET_URL, timeout=120)
    response.raise_for_status()

    with open(MICRONET_PATH, "wb") as f:
        f.write(response.content)

    print("Download complete.")
else:
    print(f"Checkpoint already exists: {MICRONET_PATH}")

# -------------------------------------------------------------------------
# 11.3 Create standard torchvision ResNet18
# -------------------------------------------------------------------------
print("Creating torchvision ResNet18...")

base_model = models.resnet18(weights=None)

# -------------------------------------------------------------------------
# 11.4 Load MicroNet checkpoint
# -------------------------------------------------------------------------
print("Loading MicroNet checkpoint...")

checkpoint = torch.load(
    MICRONET_PATH,
    map_location="cpu",
    weights_only=False
)

print(f"Checkpoint type: {type(checkpoint)}")

# The checkpoint should contain a PyTorch state_dict.
if not isinstance(checkpoint, dict):
    raise RuntimeError(
        f"Unexpected checkpoint format: {type(checkpoint)}"
    )

state_dict = checkpoint

# Remove possible DataParallel prefix.
clean_state_dict = {}

for key, value in state_dict.items():
    if key.startswith("module."):
        key = key[len("module."):]
    clean_state_dict[key] = value

# -------------------------------------------------------------------------
# 11.5 Inspect checkpoint keys BEFORE loading
# -------------------------------------------------------------------------
print("\nCheckpoint inspection")
print("-" * 80)
print(f"Number of checkpoint tensors: {len(clean_state_dict)}")

print("First 10 checkpoint keys:")
for key in list(clean_state_dict.keys())[:10]:
    print(" ", key)

# -------------------------------------------------------------------------
# 11.6 Load MicroNet checkpoint
# -------------------------------------------------------------------------
print("\nLoading MicroNet checkpoint...")

load_result = base_model.load_state_dict(
    clean_state_dict,
    strict=False
)

# PyTorch normally returns an _IncompatibleKeys object.
# Some environments may return None.
if load_result is None:
    missing_keys = []
    unexpected_keys = []
else:
    missing_keys = list(load_result.missing_keys)
    unexpected_keys = list(load_result.unexpected_keys)

print("\nCheckpoint loading result")
print("-" * 80)
print(f"Missing keys:    {len(missing_keys)}")
print(f"Unexpected keys: {len(unexpected_keys)}")

if missing_keys:
    print("\nMissing keys:")
    for key in missing_keys:
        print(" ", key)

if unexpected_keys:
    print("\nUnexpected keys:")
    for key in unexpected_keys:
        print(" ", key)

# -------------------------------------------------------------------------
# 11.7 Validate checkpoint
# -------------------------------------------------------------------------
# The MicroNet checkpoint is an encoder checkpoint.
# Therefore the ImageNet/torchvision classification head is not included.
#
# For ResNet18, the expected missing keys are exactly:
#   fc.weight
#   fc.bias
#
# These are intentionally NOT needed because we remove the fc layer below.

EXPECTED_MISSING = {
    "fc.weight",
    "fc.bias",
}

if set(missing_keys) != EXPECTED_MISSING:
    raise RuntimeError(
        "Unexpected checkpoint mismatch.\n"
        f"Expected missing keys: {EXPECTED_MISSING}\n"
        f"Actual missing keys:   {set(missing_keys)}\n"
        f"Unexpected keys:       {set(unexpected_keys)}"
    )

if unexpected_keys:
    raise RuntimeError(
        "Unexpected keys found in the MicroNet checkpoint:\n"
        + "\n".join(unexpected_keys)
    )

print("\nCheckpoint validation PASSED.")
print("The only missing parameters are the classification head (fc),")
print("which is intentionally removed for feature extraction.")

# -------------------------------------------------------------------------
# 11.8 Remove classification layer
# -------------------------------------------------------------------------
encoder = torch.nn.Sequential(
    *list(base_model.children())[:-1]
)

encoder = encoder.to(DEVICE)
encoder.eval()

# Freeze the entire encoder.
for p in encoder.parameters():
    p.requires_grad = False

print("\nEncoder configuration")
print("-" * 80)
print("Architecture:        ResNet18")
print("Pretraining:         MicroNet")
print("Classification head: removed")
print("Output embedding:    512 dimensions")
print("Backbone frozen:     YES")

# -------------------------------------------------------------------------
# 11.9 Preprocessing
# -------------------------------------------------------------------------
PREPROCESS = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

# -------------------------------------------------------------------------
# 11.10 Embedding function
# -------------------------------------------------------------------------
def embed_image(path: str) -> np.ndarray:

    try:
        with Image.open(path) as img:
            img = img.convert("RGB")
            x = PREPROCESS(img).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            feat = encoder(x)

            # ResNet18 output before classification:
            # [1, 512, 1, 1]
            vec = feat.flatten(1)

            # [1, 512] -> [512]
            vec = vec.squeeze(0).cpu().numpy()

        return vec.astype(np.float32)

    except Exception as e:
        raise RuntimeError(
            f"Could not process image {path}: {e}"
        )

# -------------------------------------------------------------------------
# 11.11 Sanity check using ONE real microscopy image
# -------------------------------------------------------------------------
test_sample = None

for block in [
    afm_3um_images,
    afm_10um_images,
    sem_cross_images,
    sem_surface_images,
]:
    for idx in EXPECTED_SAMPLES:
        if block.get(idx):
            test_sample = block[idx][0]
            break
    if test_sample:
        break

if test_sample is None:
    raise RuntimeError(
        "No microscopy image found for the MicroNet sanity check."
    )

print("\n" + "=" * 80)
print("MICRONET EMBEDDING SANITY CHECK")
print("=" * 80)

print(f"Test image: {test_sample}")

test_embedding = embed_image(test_sample)

print(f"Embedding shape:  {test_embedding.shape}")
print(f"Embedding dtype:  {test_embedding.dtype}")
print(f"All finite:       {np.isfinite(test_embedding).all()}")
print(f"Min:              {test_embedding.min():.6f}")
print(f"Max:              {test_embedding.max():.6f}")
print(f"Mean:             {test_embedding.mean():.6f}")
print(f"Std:              {test_embedding.std():.6f}")

# -------------------------------------------------------------------------
# 11.12 Final validation
# -------------------------------------------------------------------------
if test_embedding.shape != (512,):
    raise RuntimeError(
        f"Expected a 512-dimensional embedding, "
        f"but received {test_embedding.shape}"
    )

if not np.isfinite(test_embedding).all():
    raise RuntimeError(
        "Embedding contains NaN or Inf values."
    )

if np.allclose(test_embedding, 0):
    raise RuntimeError(
        "Embedding is effectively all zeros."
    )

print("\n" + "=" * 80)
print("SUCCESS")
print("=" * 80)
print("MicroNet ResNet18 is loaded and frozen.")
print("Classification head is excluded.")
print("512-dimensional embeddings are ready.")


LOADING MICRONET-PRETRAINED RESNET18 (frozen)
Device: cpu
Download complete.
Creating torchvision ResNet18...
Loading MicroNet checkpoint...
Checkpoint type: <class 'dict'>

Checkpoint inspection
--------------------------------------------------------------------------------
Number of checkpoint tensors: 120
First 10 checkpoint keys:
  conv1.weight
  bn1.weight
  bn1.bias
  bn1.running_mean
  bn1.running_var
  bn1.num_batches_tracked
  layer1.0.conv1.weight
  layer1.0.bn1.weight
  layer1.0.bn1.bias
  layer1.0.bn1.running_mean

Loading MicroNet checkpoint...

Checkpoint loading result
--------------------------------------------------------------------------------
Missing keys:    2
Unexpected keys: 0

Missing keys:
  fc.weight
  fc.bias

Checkpoint validation PASSED.
The only missing parameters are the classification head (fc),
which is intentionally removed for feature extraction.

Encoder configuration
--------------------------------------------------------------------------------

In [12]:
# =============================================================================
# 12. BUILD PER-SAMPLE EMBEDDINGS (each block kept SEPARATE, not pooled)
# =============================================================================
def build_embeddings(image_dict, block_name):
    print(f"\nExtracting embeddings: {block_name}")
    rows = []
    for idx in EXPECTED_SAMPLES:
        paths = image_dict[idx]
        vecs = np.stack([embed_image(p) for p in paths])
        mags = sorted({parse_magnification(p) for p in paths if parse_magnification(p)})
        rows.append({"Sample": f"S{idx}", "n_images": len(paths),
                      "magnifications": ",".join(mags) if mags else "n/a",
                      "embedding": vecs.mean(axis=0)})
        print(f"  S{idx:02d}: {len(paths)} image(s) -> averaged")
    df = pd.DataFrame(rows).set_index("Sample")
    print(f"  {block_name}: {len(df)} samples, embedding dim = {len(df.iloc[0]['embedding'])}")
    return df

afm3_emb  = build_embeddings(afm_3um_images,     "AFM_3x3")
afm10_emb = build_embeddings(afm_10um_images,    "AFM_10x10")
cross_emb = build_embeddings(sem_cross_images,   "SEM_cross")
surf_emb  = build_embeddings(sem_surface_images, "SEM_surface")

SAMPLE_ORDER = [f"S{i}" for i in EXPECTED_SAMPLES]

def align(df):
    return np.stack([df.loc[s, "embedding"] for s in SAMPLE_ORDER])

AFM3_X, AFM10_X = align(afm3_emb), align(afm10_emb)
CROSS_X, SURF_X = align(cross_emb), align(surf_emb)

image_counts = pd.DataFrame({
    "Sample": SAMPLE_ORDER,
    "AFM_3x3_images":       [afm3_emb.loc[s, "n_images"] for s in SAMPLE_ORDER],
    "AFM_10x10_images":     [afm10_emb.loc[s, "n_images"] for s in SAMPLE_ORDER],
    "SEM_cross_images":     [cross_emb.loc[s, "n_images"] for s in SAMPLE_ORDER],
    "SEM_cross_mags":       [cross_emb.loc[s, "magnifications"] for s in SAMPLE_ORDER],
    "SEM_surface_images":   [surf_emb.loc[s, "n_images"] for s in SAMPLE_ORDER],
    "SEM_surface_mags":     [surf_emb.loc[s, "magnifications"] for s in SAMPLE_ORDER],
})
print("\nIMAGE COUNTS PER SAMPLE (magnification lists kept for future secondary analysis)")
print(image_counts.to_string(index=False))
image_counts.to_csv("image_counts_per_sample.csv", index=False)


Extracting embeddings: AFM_3x3
  S00: 1 image(s) -> averaged
  S01: 1 image(s) -> averaged
  S02: 1 image(s) -> averaged
  S03: 1 image(s) -> averaged
  S04: 1 image(s) -> averaged
  S05: 1 image(s) -> averaged
  S06: 1 image(s) -> averaged
  S07: 1 image(s) -> averaged
  S08: 1 image(s) -> averaged
  S09: 1 image(s) -> averaged
  S10: 1 image(s) -> averaged
  S11: 1 image(s) -> averaged
  S12: 1 image(s) -> averaged
  S13: 1 image(s) -> averaged
  S14: 1 image(s) -> averaged
  S15: 1 image(s) -> averaged
  S16: 1 image(s) -> averaged
  S17: 1 image(s) -> averaged
  S18: 1 image(s) -> averaged
  S19: 1 image(s) -> averaged
  S20: 1 image(s) -> averaged
  S21: 1 image(s) -> averaged
  S22: 1 image(s) -> averaged
  S23: 1 image(s) -> averaged
  S24: 1 image(s) -> averaged
  S25: 1 image(s) -> averaged
  S26: 1 image(s) -> averaged
  S27: 1 image(s) -> averaged
  AFM_3x3: 28 samples, embedding dim = 512

Extracting embeddings: AFM_10x10
  S00: 1 image(s) -> averaged
  S01: 1 image(s) -> 

In [13]:
# =============================================================================
# 13. LOOCV WITH PCA FIT STRICTLY INSIDE EACH FOLD
# =============================================================================
def loocv_predict_multimodal(y, model, tabular_X=None, image_blocks=None, n_pca=5):
    n = len(y)
    preds = np.zeros(n)
    for i in range(n):
        train_idx = [j for j in range(n) if j != i]
        parts_train, parts_test = [], []

        if tabular_X is not None:
            scaler = MinMaxScaler()
            parts_train.append(scaler.fit_transform(tabular_X.iloc[train_idx]))
            parts_test.append(scaler.transform(tabular_X.iloc[[i]]))

        if image_blocks:
            for mat in image_blocks.values():
                k = min(n_pca, len(train_idx), mat.shape[1])
                pca = PCA(n_components=k, random_state=RANDOM_STATE)
                parts_train.append(pca.fit_transform(mat[train_idx]))
                parts_test.append(pca.transform(mat[[i]]))

        X_train, X_test = np.hstack(parts_train), np.hstack(parts_test)
        m = clone(model)
        m.fit(X_train, y.iloc[train_idx])
        preds[i] = m.predict(X_test)[0]
    return preds

def evaluate_experiment(name, tabular_X=None, image_blocks=None, n_pca=5):
    rows = []
    for model_name, model in build_models().items():
        preds = loocv_predict_multimodal(y, model, tabular_X, image_blocks, n_pca)
        rows.append({"Feature_set": name, "Model": model_name,
                      "MAPE": mean_absolute_percentage_error(y, preds),
                      "RMSE": mean_squared_error(y, preds) ** 0.5,
                      "R2": r2_score(y, preds)})
    return pd.DataFrame(rows)

In [14]:
# =============================================================================
# 14. FEATURE-SET EXPERIMENTS
# =============================================================================
EXPERIMENTS = {
    "FS1_tabular_only":     dict(tabular_X=ml_data[feature_sets["FS1_process_only"]], image_blocks=None),
    "FS2_tabular_only":     dict(tabular_X=ml_data[feature_sets["FS2_process_plus_SFE"]], image_blocks=None),
    "FS3_tabular_only":     dict(tabular_X=X_tab, image_blocks=None),
    "FS4_tabular_only":     dict(tabular_X=ml_data[feature_sets["FS4_process_plus_Ra_VD_SFE"]], image_blocks=None),
    "AFM_3x3_only":         dict(tabular_X=None,  image_blocks={"AFM_3x3": AFM3_X}),
    "AFM_10x10_only":       dict(tabular_X=None,  image_blocks={"AFM_10x10": AFM10_X}),
    "AFM_combined_only":    dict(tabular_X=None,  image_blocks={"AFM_3x3": AFM3_X, "AFM_10x10": AFM10_X}),
    "SEM_cross_only":       dict(tabular_X=None,  image_blocks={"SEM_cross": CROSS_X}),
    "SEM_surface_only":     dict(tabular_X=None,  image_blocks={"SEM_surface": SURF_X}),
    "SEM_combined_only":    dict(tabular_X=None,  image_blocks={"SEM_cross": CROSS_X, "SEM_surface": SURF_X}),
    "all_images_only":      dict(tabular_X=None,  image_blocks={"AFM_3x3": AFM3_X, "AFM_10x10": AFM10_X,
                                                                  "SEM_cross": CROSS_X, "SEM_surface": SURF_X}),
    "tabular_plus_AFM":     dict(tabular_X=X_tab, image_blocks={"AFM_3x3": AFM3_X, "AFM_10x10": AFM10_X}),
    "tabular_plus_SEM":     dict(tabular_X=X_tab, image_blocks={"SEM_cross": CROSS_X, "SEM_surface": SURF_X}),
    "tabular_plus_allimages": dict(tabular_X=X_tab, image_blocks={"AFM_3x3": AFM3_X, "AFM_10x10": AFM10_X,
                                                                    "SEM_cross": CROSS_X, "SEM_surface": SURF_X}),
}

In [15]:
# =============================================================================
# 15. PCA SENSITIVITY - report every value, never cherry-pick after the fact
# =============================================================================
PCA_GRID = [2, 3, 5]
print("\n" + "=" * 80)
print("PCA SENSITIVITY (all values reported)")
print("=" * 80)
sens_rows = []
for n_pca in PCA_GRID:
    for exp_name, cfg in EXPERIMENTS.items():
        if cfg["image_blocks"] is None:
            continue
        df_res = evaluate_experiment(exp_name, n_pca=n_pca, **cfg)
        best = df_res.loc[df_res["MAPE"].idxmin()]
        sens_rows.append({"Feature_set": exp_name, "n_pca": n_pca,
                           "best_model": best["Model"], "MAPE": best["MAPE"], "R2": best["R2"]})
sensitivity_df = pd.DataFrame(sens_rows)
print(sensitivity_df.pivot(index="Feature_set", columns="n_pca", values="MAPE").round(4))
sensitivity_df.to_csv("pca_sensitivity_results.csv", index=False)
print("\nIf a feature set's rank swings a lot across n_pca columns, treat its 'best'")
print("result there as noise, not a real effect.")


PCA SENSITIVITY (all values reported)
n_pca                        2       3       5
Feature_set                                   
AFM_10x10_only          0.1564  0.1739  0.1801
AFM_3x3_only            0.1655  0.1791  0.1891
AFM_combined_only       0.1704  0.1834  0.1791
SEM_combined_only       0.1729  0.1708  0.1699
SEM_cross_only          0.1707  0.1565  0.1682
SEM_surface_only        0.1696  0.1728  0.1510
all_images_only         0.1685  0.1767  0.1691
tabular_plus_AFM        0.1630  0.1731  0.1737
tabular_plus_SEM        0.1612  0.1553  0.1679
tabular_plus_allimages  0.1663  0.1726  0.1607

If a feature set's rank swings a lot across n_pca columns, treat its 'best'
result there as noise, not a real effect.


In [16]:
# =============================================================================
# 16. PRIMARY RUN (n_pca=5, chosen a priori per the ~5-component plan agreed
#     with Dr. Ahmed - PCA=2/3 above are the sensitivity checks, not the main
#     result)
# =============================================================================
OFFICIAL_PCA = 5
print("\n" + "=" * 80)
print(f"PRIMARY RESULTS - n_pca={OFFICIAL_PCA}")
print("=" * 80)
all_rows = []
for exp_name, cfg in EXPERIMENTS.items():
    df_res = evaluate_experiment(exp_name, n_pca=OFFICIAL_PCA, **cfg)
    all_rows.append(df_res)
final_df = pd.concat(all_rows, ignore_index=True)
final_df["MAPE_percent"] = final_df["MAPE"] * 100

print("\nMAPE (%):")
print(final_df.pivot(index="Model", columns="Feature_set", values="MAPE_percent").round(2))
print("\nR2:")
print(final_df.pivot(index="Model", columns="Feature_set", values="R2").round(3))
print("\nRMSE (MPa):")
print(final_df.pivot(index="Model", columns="Feature_set", values="RMSE").round(4))


PRIMARY RESULTS - n_pca=5

MAPE (%):
Feature_set  AFM_10x10_only  AFM_3x3_only  AFM_combined_only  \
Model                                                          
AB                    20.43         22.22              21.91   
BG                    20.16         21.50              20.03   
DT                    25.88         27.81              30.40   
ET                    18.91         19.13              17.91   
GB                    21.15         27.12              24.25   
KNN                   19.68         18.91              20.77   
RF                    19.06         19.71              18.59   
SVR                   18.01         21.10              18.19   

Feature_set  FS1_tabular_only  FS2_tabular_only  FS3_tabular_only  \
Model                                                               
AB                      13.16             16.35             16.86   
BG                      15.08             14.78             15.95   
DT                      13.53             17.

In [17]:
# =============================================================================
# 17. EXPLICIT FS3-vs-MULTIMODAL DELTA (same model, same PCA, side by side)
# =============================================================================
print("\n" + "=" * 80)
print("FS3 (tabular only) vs each multimodal combination - same model, same PCA")
print("=" * 80)
delta_rows = []
for model_name in build_models().keys():
    fs3_row = final_df[(final_df.Feature_set == "FS3_tabular_only") & (final_df.Model == model_name)].iloc[0]
    for combo in ["tabular_plus_AFM", "tabular_plus_SEM", "tabular_plus_allimages"]:
        combo_row = final_df[(final_df.Feature_set == combo) & (final_df.Model == model_name)].iloc[0]
        delta_rows.append({
            "Model": model_name, "Combination": combo,
            "FS3_MAPE_%": fs3_row["MAPE"] * 100,
            "Combo_MAPE_%": combo_row["MAPE"] * 100,
            "Delta_MAPE_pp": (combo_row["MAPE"] - fs3_row["MAPE"]) * 100,
            "FS3_R2": fs3_row["R2"], "Combo_R2": combo_row["R2"],
        })
delta_df = pd.DataFrame(delta_rows)
print(delta_df.round(3).to_string(index=False))
delta_df.to_csv("tabular_vs_multimodal_delta.csv", index=False)


FS3 (tabular only) vs each multimodal combination - same model, same PCA
Model            Combination  FS3_MAPE_%  Combo_MAPE_%  Delta_MAPE_pp  FS3_R2  Combo_R2
   DT       tabular_plus_AFM      16.273        28.690         12.418  -0.161    -1.769
   DT       tabular_plus_SEM      16.273        25.868          9.596  -0.161    -1.180
   DT tabular_plus_allimages      16.273        28.523         12.251  -0.161    -1.859
   RF       tabular_plus_AFM      16.571        18.377          1.806   0.001    -0.121
   RF       tabular_plus_SEM      16.571        17.074          0.503   0.001     0.005
   RF tabular_plus_allimages      16.571        17.348          0.777   0.001    -0.034
   ET       tabular_plus_AFM      15.864        17.372          1.508   0.126    -0.042
   ET       tabular_plus_SEM      15.864        16.793          0.928   0.126     0.007
   ET tabular_plus_allimages      15.864        17.381          1.516   0.126    -0.050
   BG       tabular_plus_AFM      15.953      

In [18]:
# =============================================================================
# 18. SUMMARY
# =============================================================================
def best_in(fs_list):
    subset = final_df[final_df["Feature_set"].isin(fs_list)]
    return subset.loc[subset["MAPE"].idxmin()]

fs3_best = best_in(["FS3_tabular_only"])
image_only_best = best_in(["AFM_3x3_only", "AFM_10x10_only", "AFM_combined_only",
                            "SEM_cross_only", "SEM_surface_only", "SEM_combined_only", "all_images_only"])
multimodal_best = best_in(["tabular_plus_AFM", "tabular_plus_SEM", "tabular_plus_allimages"])

print("\n" + "=" * 80)
print("FINAL SUMMARY")
print("=" * 80)
print(f"Tabular only (FS3):    {fs3_best['Model']:>4} | MAPE={fs3_best['MAPE']*100:.2f}% | R2={fs3_best['R2']:.3f}")
print(f"Best image-only:       {image_only_best['Model']:>4} / {image_only_best['Feature_set']} "
      f"| MAPE={image_only_best['MAPE']*100:.2f}% | R2={image_only_best['R2']:.3f}")
print(f"Best multimodal:       {multimodal_best['Model']:>4} / {multimodal_best['Feature_set']} "
      f"| MAPE={multimodal_best['MAPE']*100:.2f}% | R2={multimodal_best['R2']:.3f}")
print("\nCheck the FS3-vs-multimodal delta table and the PCA-sensitivity table")
print("above before concluding images help - at N=28 a win that disappears when")
print("n_pca changes, or that is smaller than a couple of MAPE points, is noise.")
print("\nEXCLUDED FROM THIS PIPELINE: SEM_top_view_failure (target leakage) and")
print("any '_cropped'/'_highlight' SEM variant files (edited copies, not independent images).")


FINAL SUMMARY
Tabular only (FS3):     KNN | MAPE=15.53% | R2=0.042
Best image-only:         AB / SEM_surface_only | MAPE=15.10% | R2=0.084
Best multimodal:        KNN / tabular_plus_allimages | MAPE=16.07% | R2=0.018

Check the FS3-vs-multimodal delta table and the PCA-sensitivity table
above before concluding images help - at N=28 a win that disappears when
n_pca changes, or that is smaller than a couple of MAPE points, is noise.

EXCLUDED FROM THIS PIPELINE: SEM_top_view_failure (target leakage) and
any '_cropped'/'_highlight' SEM variant files (edited copies, not independent images).


In [19]:
# =============================================================================
# 19. SAVE
# =============================================================================
final_df.to_csv("full_multimodal_results.csv", index=False)
print("\nSaved: full_multimodal_results.csv, pca_sensitivity_results.csv,")
print("tabular_vs_multimodal_delta.csv, image_counts_per_sample.csv")
print("Pipeline completed.")


Saved: full_multimodal_results.csv, pca_sensitivity_results.csv,
tabular_vs_multimodal_delta.csv, image_counts_per_sample.csv
Pipeline completed.


In [20]:
# =============================================================================
# 20. DO THE PREDICTIONS LAND INSIDE THE REPLICATE RANGE?
#     For the best tabular-only (FS3) and best multimodal model, check every
#     LOOCV prediction against the min..max of the 3 measured replicates
#     (and a tolerance band of 5 % around that range).
# =============================================================================
REP_PREFIX = {"SLSS_MPa": "SLSS", "FractureEnergy_mJmm3": "FE"}[TARGET]
REP_SCALE = FE_UNIT_FACTOR if TARGET == "FractureEnergy_mJmm3" else 1.0
rep_all = rep_cols(REP_PREFIX).loc[ml_data.index] * REP_SCALE
rep_min, rep_max = rep_all.min(axis=1), rep_all.max(axis=1)

def range_report(label, feature_set, model_name):
    cfg = EXPERIMENTS[feature_set]
    preds = loocv_predict_multimodal(y, build_models()[model_name], n_pca=OFFICIAL_PCA, **cfg)
    inside = (preds >= rep_min.values) & (preds <= rep_max.values)
    near = (preds >= rep_min.values * 0.95) & (preds <= rep_max.values * 1.05)
    out = pd.DataFrame({"Sample": ml_data.index, "Measured_mean": y.values,
                        "Rep_min": rep_min.values, "Rep_max": rep_max.values,
                        "Predicted": preds, "Inside_range": inside,
                        "Within_range_pm5pct": near})
    print(f"\n{label}: {feature_set} / {model_name}")
    print(f"  inside replicate range: {inside.sum()}/{len(inside)}   "
          f"within range +-5%: {near.sum()}/{len(near)}")
    return out

rr_fs3 = range_report("Tabular only", "FS3_tabular_only", fs3_best["Model"])
rr_mm = range_report("Best multimodal", multimodal_best["Feature_set"], multimodal_best["Model"])
rr_fs3.to_csv("predictions_vs_replicates_FS3.csv", index=False)
rr_mm.to_csv("predictions_vs_replicates_multimodal.csv", index=False)


Tabular only: FS3_tabular_only / KNN
  inside replicate range: 10/28   within range +-5%: 13/28

Best multimodal: tabular_plus_allimages / KNN
  inside replicate range: 9/28   within range +-5%: 13/28


In [21]:
# =============================================================================
# 21. EXPORT THE TABULAR DATASET USED (replicate means + replicate min/max)
# =============================================================================
export = data.copy()
for prefix, col, k in [("Ra", "Roughness_nm", 1), ("VD", "ValleyDepth_nm", 1),
                       ("SLSS", "SLSS_MPa", 1), ("FE", "FractureEnergy_mJmm3", FE_UNIT_FACTOR)]:
    export[f"{col}_rep_min"] = rep_cols(prefix).min(axis=1) * k
    export[f"{col}_rep_max"] = rep_cols(prefix).max(axis=1) * k
export.to_csv("tabular_dataset_S0_S27.csv")
print("\nSaved: predictions_vs_replicates_FS3.csv, predictions_vs_replicates_multimodal.csv,")
print("tabular_dataset_S0_S27.csv")


Saved: predictions_vs_replicates_FS3.csv, predictions_vs_replicates_multimodal.csv,
tabular_dataset_S0_S27.csv
